# Number Systems for AI/ML

## Why Number Systems Matter in Machine Learning

Understanding number systems is **fundamental** to AI/ML engineering. Every aspect of machine learning—from data representation to model weights to GPU computations—relies on how numbers are stored and manipulated.

### Key Topics Covered

| Topic | ML Application |
|-------|----------------|
| Number Types | Memory optimization, mixed precision training |
| Floating Point | Gradient stability, loss computation |
| Binary/Hex | Bit manipulation, efficient storage |
| Complex Numbers | Fourier transforms, signal processing |
| Quantization | Model compression, edge deployment |

### Prerequisites
- Basic Python programming
- NumPy fundamentals

In [ ]:
# Core Libraries for Number System Examples
import numpy as np
import warnings

# Display settings
np.set_printoptions(precision=6, suppress=True)

# Version check
print(f"NumPy version: {np.__version__}")
print("Setup complete! Ready to explore number systems.")

---

## 1. Number Types in Python and NumPy

In ML, choosing the right number type affects:
- **Memory usage**: float16 uses 2 bytes vs float64's 8 bytes
- **Computation speed**: Smaller types = faster GPU operations  
- **Precision**: Trade-off between accuracy and efficiency

### The Number Hierarchy

```
ℂ (Complex) ⊃ ℝ (Real) ⊃ ℚ (Rational) ⊃ ℤ (Integer) ⊃ ℕ (Natural)
```

| Symbol | Set | Python Type | ML Use Case |
|--------|-----|-------------|-------------|
| ℕ | Natural (0,1,2...) | `int` | Indices, counts |
| ℤ | Integers | `int` | Token IDs, embeddings |
| ℚ | Rationals | `float` | Learning rates |
| ℝ | Reals | `float32/64` | Weights, losses |
| ℂ | Complex | `complex` | FFT, eigenvalues |

In [ ]:
# Python Native Number Types

# Integers - Arbitrary precision (no overflow!)
big_int = 10 ** 100  # Googol - Python handles this natively
print(f"Big integer (10^100): {str(big_int)[:30]}... ({len(str(big_int))} digits)")
print(f"Type: {type(big_int)}")

# Float - Double precision (64-bit) by default
pi = 3.141592653589793
print(f"\nFloat (π): {pi}")
print(f"Type: {type(pi)}")

# Complex - Used in signal processing, FFT
z = 3 + 4j
print(f"\nComplex: {z}")
print(f"  Real: {z.real}, Imaginary: {z.imag}")
print(f"  Magnitude |z|: {abs(z)}")  # √(3² + 4²) = 5

In [ ]:
# NumPy Data Types - Critical for ML Memory Management

print("NumPy Integer Types:")
print(f"{'Type':<12} {'Bytes':<8} {'Min':<25} {'Max':<25}")
print("-" * 70)

for dtype in [np.int8, np.int16, np.int32, np.int64]:
    info = np.iinfo(dtype)
    print(f"{dtype.__name__:<12} {dtype().nbytes:<8} {info.min:<25} {info.max:<25}")

print("\nNumPy Float Types:")
print(f"{'Type':<12} {'Bytes':<8} {'Precision':<15} {'ML Use Case':<30}")
print("-" * 70)

float_info = [
    (np.float16, "~3 digits", "Inference, mixed precision"),
    (np.float32, "~7 digits", "Standard training (default)"),
    (np.float64, "~15 digits", "High precision, scientific"),
]

for dtype, prec, use in float_info:
    print(f"{dtype.__name__:<12} {dtype().nbytes:<8} {prec:<15} {use:<30}")

---

## 2. Floating Point Precision Issues

Floating point numbers are **approximations** due to binary representation. This causes issues in:

- **Gradient accumulation**: Small errors compound over iterations
- **Loss comparisons**: Direct equality checks fail
- **Financial/scientific ML**: Precision errors accumulate

### IEEE 754 Standard

A float64 stores: `(-1)^sign × 1.mantissa × 2^(exponent-1023)`

```
64-bit float layout:
┌───┬──────────────┬────────────────────────────────────────────────────────┐
│ S │  Exponent    │                    Mantissa (52 bits)                  │
│1b │   11 bits    │                                                        │
└───┴──────────────┴────────────────────────────────────────────────────────┘
```

In [ ]:
# Classic Floating Point Precision Issue

print("The Famous 0.1 + 0.2 Problem:")
print("-" * 40)
result = 0.1 + 0.2
print(f"0.1 + 0.2 = {result}")
print(f"0.1 + 0.2 == 0.3? {result == 0.3}")  # False!
print(f"Difference: {result - 0.3:.2e}")

# Solution: Use np.isclose() or np.allclose()
print(f"\nUsing np.isclose(): {np.isclose(result, 0.3)}")  # True

# Why? 0.1 in binary is repeating: 0.0001100110011...
print(f"\n0.1 stored as: {0.1:.20f}")
print(f"0.2 stored as: {0.2:.20f}")
print(f"0.3 stored as: {0.3:.20f}")

In [ ]:
# Machine Epsilon - Smallest distinguishable difference from 1.0
# Critical for numerical stability in gradient computations

print("Machine Epsilon (ε) by dtype:")
print("-" * 50)

for dtype in [np.float16, np.float32, np.float64]:
    eps = np.finfo(dtype).eps
    one = dtype(1.0)
    
    # Verify: 1.0 + ε ≠ 1.0, but 1.0 + ε/2 = 1.0
    test1 = (one + dtype(eps)) != one
    test2 = (one + dtype(eps / 2)) == one
    
    print(f"{dtype.__name__:<10} ε = {eps:<12.2e} | 1+ε ≠ 1: {test1}, 1+ε/2 = 1: {test2}")

# ML Implication: Learning rates smaller than ε have no effect!
print(f"\n⚠️ For float32: learning_rate < {np.finfo(np.float32).eps:.0e} is effectively zero")

---

## 3. Numerical Overflow and Underflow

Two critical issues in neural network training:

| Issue | Description | Common Cause | Solution |
|-------|-------------|--------------|----------|
| **Overflow** | Number too large → `inf` | Large logits in softmax | Subtract max value |
| **Underflow** | Number too small → `0` | Product of probabilities | Use log-space |

### The Softmax Problem

Softmax: $\sigma(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$

When $x_i$ is large, $e^{x_i} \to \infty$ (overflow!)

**Solution**: Subtract max: $\sigma(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$

In [ ]:
# Softmax Overflow Problem and Solution

def naive_softmax(x):
    """Prone to overflow with large values."""
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

def stable_softmax(x):
    """Numerically stable version - used in all frameworks."""
    exp_x = np.exp(x - np.max(x))  # Subtract max for stability
    return exp_x / np.sum(exp_x)

# Test with normal values
x_normal = np.array([1.0, 2.0, 3.0])
print("Normal case:")
print(f"  Input: {x_normal}")
print(f"  Naive:  {naive_softmax(x_normal)}")
print(f"  Stable: {stable_softmax(x_normal)}")

# Test with large values (simulates large neural network logits)
x_large = np.array([1000.0, 1001.0, 1002.0])
print(f"\nLarge logits (overflow case):")
print(f"  Input: {x_large}")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(f"  Naive:  {naive_softmax(x_large)}")  # NaN!
print(f"  Stable: {stable_softmax(x_large)}")  # Works!

In [ ]:
# Underflow Problem: Product of Small Probabilities

# Scenario: Computing probability of a sequence (e.g., language model)
# P(sentence) = P(w1) × P(w2|w1) × P(w3|w1,w2) × ...

small_probs = np.array([0.001] * 100)  # 100 words, each with P=0.001

# Direct product - underflows to zero!
direct_product = np.prod(small_probs)
print("Underflow in probability computation:")
print(f"  Product of 100 × 0.001: {direct_product}")  # 0.0!

# Solution: Work in log space
# log(P) = log(p1) + log(p2) + ... + log(pn)
log_prob = np.sum(np.log(small_probs))
print(f"  Sum of log probabilities: {log_prob:.2f}")
print(f"  This represents: e^{log_prob:.0f} (tracked without underflow)")

# To compare probabilities, just compare log values!
# Higher log = higher probability

In [ ]:
# Log-Sum-Exp Trick - Essential for Cross-Entropy Loss
# Computes: log(sum(exp(x))) stably

def logsumexp_naive(x):
    """Unstable - overflows with large values."""
    return np.log(np.sum(np.exp(x)))

def logsumexp_stable(x):
    """Stable version: log(Σexp(x)) = max(x) + log(Σexp(x - max(x)))"""
    max_x = np.max(x)
    return max_x + np.log(np.sum(np.exp(x - max_x)))

x = np.array([1000, 1001, 1002])
print("Log-Sum-Exp Trick:")
print(f"  Input: {x}")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(f"  Naive:  {logsumexp_naive(x)}")  # inf
print(f"  Stable: {logsumexp_stable(x):.4f}")

# scipy has this built-in: scipy.special.logsumexp
from scipy.special import logsumexp
print(f"  SciPy:  {logsumexp(x):.4f}")

---

## 4. Complex Numbers in Machine Learning

Complex numbers: $z = a + bi$ where $i^2 = -1$

### ML Applications

| Application | Usage |
|-------------|-------|
| **Fourier Transform** | Audio/image processing, CNNs |
| **Eigenvalues** | PCA, spectral clustering |
| **Signal Processing** | Speech recognition, time series |
| **Quantum ML** | Quantum computing algorithms |

### Key Properties

- Magnitude: $|z| = \sqrt{a^2 + b^2}$  
- Phase: $\theta = \arctan(b/a)$  
- Euler's formula: $e^{i\theta} = \cos\theta + i\sin\theta$

In [ ]:
# Complex Number Operations in Python

z1 = 3 + 4j  # Pythagorean triple: |z| = 5
z2 = 1 - 2j

print("Complex Number Operations:")
print(f"  z1 = {z1}")
print(f"  z2 = {z2}")
print(f"  Real(z1) = {z1.real}, Imag(z1) = {z1.imag}")

# Operations
print(f"\n  z1 + z2 = {z1 + z2}")
print(f"  z1 × z2 = {z1 * z2}")
print(f"  |z1| (magnitude) = {abs(z1)}")  # sqrt(3² + 4²) = 5
print(f"  z1* (conjugate) = {z1.conjugate()}")

# Important identity: z × z* = |z|²  (always real!)
print(f"  z1 × z1* = {z1 * z1.conjugate()}")  # 25 + 0j

# Euler's formula: e^(iπ) = -1
euler = np.exp(1j * np.pi)
print(f"\n  e^(iπ) = {euler.real:.0f} + {euler.imag:.2e}i ≈ -1")

In [ ]:
# Fourier Transform - Where Complex Numbers Shine in ML
# Used in: audio processing, image filtering, CNNs

# Create a simple signal: sum of two frequencies
t = np.linspace(0, 1, 256)  # 1 second, 256 samples
freq1, freq2 = 5, 20  # Hz
signal = np.sin(2 * np.pi * freq1 * t) + 0.5 * np.sin(2 * np.pi * freq2 * t)

# Apply FFT - result is complex!
fft_result = np.fft.fft(signal)

print("Fast Fourier Transform (FFT):")
print(f"  Input signal shape: {signal.shape}")
print(f"  FFT result shape: {fft_result.shape}")
print(f"  FFT dtype: {fft_result.dtype}")  # complex128

# Magnitude spectrum reveals frequencies
frequencies = np.fft.fftfreq(len(signal), t[1] - t[0])
magnitude = np.abs(fft_result)

# Find peaks (dominant frequencies)
peak_indices = np.argsort(magnitude)[-4:-1][::-1]  # Top 3 excluding DC
print(f"\n  Detected frequencies: {frequencies[peak_indices[:2]].astype(int)} Hz")
print(f"  (Expected: {freq1} Hz and {freq2} Hz)")

---

## 5. Quantization for ML Model Deployment

**Quantization** = Converting float32 weights to lower precision (int8, int4)

### Why Quantize?

| Metric | Float32 | Int8 | Improvement |
|--------|---------|------|-------------|
| Memory | 4 bytes | 1 byte | **4× smaller** |
| Speed | Baseline | Faster | **2-4× faster** |
| Accuracy | 100% | ~99% | Small loss |

### Types of Quantization

1. **Post-Training Quantization (PTQ)**: Quantize after training
2. **Quantization-Aware Training (QAT)**: Train with quantization in mind
3. **Dynamic Quantization**: Quantize at runtime

In [ ]:
# Basic Quantization: Float32 → Int8

def quantize_to_int8(weights, symmetric=True):
    """
    Quantize float32 weights to int8.
    Scale = max(|weights|) / 127
    """
    scale = np.max(np.abs(weights)) / 127
    quantized = np.round(weights / scale).astype(np.int8)
    return quantized, scale

def dequantize_from_int8(quantized, scale):
    """Convert int8 back to float32."""
    return quantized.astype(np.float32) * scale

# Simulate model weights (normally distributed)
np.random.seed(42)
weights = np.random.randn(1000).astype(np.float32)

# Quantize
q_weights, scale = quantize_to_int8(weights)
reconstructed = dequantize_from_int8(q_weights, scale)

print("Quantization Example:")
print(f"  Original dtype: {weights.dtype}, size: {weights.nbytes} bytes")
print(f"  Quantized dtype: {q_weights.dtype}, size: {q_weights.nbytes} bytes")
print(f"  Compression: {weights.nbytes / q_weights.nbytes:.1f}×")
print(f"  Quantization error (MSE): {np.mean((weights - reconstructed)**2):.6f}")

---

## 6. Number Bases and Binary Operations

Understanding binary is essential for:
- **Memory layout** of tensors
- **Bitwise tricks** for optimization
- **Hash functions** in feature engineering
- **Efficient storage** of sparse data

### Common Number Bases

| Base | Name | Digits | Use Case |
|------|------|--------|----------|
| 2 | Binary | 0, 1 | Computer storage |
| 8 | Octal | 0-7 | Unix permissions |
| 10 | Decimal | 0-9 | Human-readable |
| 16 | Hexadecimal | 0-F | Memory addresses, colors |

In [ ]:
# Number Base Conversions in Python

n = 255  # Example number

print(f"Number: {n}")
print(f"  Binary:      {bin(n)} = {n:08b}")
print(f"  Octal:       {oct(n)}")
print(f"  Hexadecimal: {hex(n)}")

# Converting FROM different bases
print("\nParsing from different bases:")
print(f"  int('11111111', 2) = {int('11111111', 2)}")   # Binary to decimal
print(f"  int('377', 8)      = {int('377', 8)}")       # Octal to decimal
print(f"  int('FF', 16)      = {int('FF', 16)}")       # Hex to decimal

# Understanding binary representation
print("\nBinary breakdown of 255:")
for i in range(7, -1, -1):
    bit = (n >> i) & 1
    print(f"  2^{i} × {bit} = {(2**i) * bit:3d}")

In [ ]:
# Bitwise Operations - Used in Efficient ML Implementations

a = 0b1100  # 12
b = 0b1010  # 10

print(f"a = {a:04b} ({a})")
print(f"b = {b:04b} ({b})")
print("-" * 30)
print(f"a & b  (AND)  = {a & b:04b} ({a & b})")
print(f"a | b  (OR)   = {a | b:04b} ({a | b})")
print(f"a ^ b  (XOR)  = {a ^ b:04b} ({a ^ b})")
print(f"~a     (NOT)  = {~a & 0xF:04b} ({~a & 0xF})")  # Masked to 4 bits
print(f"a << 1 (LEFT) = {a << 1:05b} ({a << 1})")  # Multiply by 2
print(f"a >> 1 (RIGHT)= {a >> 1:04b} ({a >> 1})")  # Divide by 2

# ML Application: Efficient power of 2 checks
def is_power_of_2(n):
    """Used in batch size validation."""
    return n > 0 and (n & (n - 1)) == 0

print(f"\nBatch size validation (power of 2):")
for batch in [16, 32, 48, 64, 128]:
    print(f"  {batch}: {'✓' if is_power_of_2(batch) else '✗'}")

---

## 7. Mixed Precision Training

Modern deep learning uses **mixed precision** to combine:
- **Float16/BFloat16**: Fast forward/backward pass
- **Float32**: Master weights, loss scaling

### Benefits

| Aspect | Float32 Only | Mixed Precision |
|--------|--------------|-----------------|
| Memory | Baseline | ~50% reduction |
| Speed | Baseline | 2-3× faster |
| Accuracy | 100% | ~Same |

### The BFloat16 Format

```
Float16:  1 sign + 5 exp + 10 mantissa (more precision)
BFloat16: 1 sign + 8 exp + 7 mantissa  (same range as float32)
```

BFloat16 preferred in ML because it handles the same range of values as float32.

In [ ]:
# Simulating Mixed Precision Training Concepts

# Compare float16 vs float32 precision
print("Float16 vs Float32 Precision:")
print("-" * 40)

# Small gradients - float16 might underflow
small_grad = np.float32(1e-5)
print(f"Small gradient (1e-5):")
print(f"  float32: {small_grad}")
print(f"  float16: {np.float16(small_grad)}")

# Very small gradient - float16 underflows!
tiny_grad = np.float32(1e-6)
print(f"\nTiny gradient (1e-6):")
print(f"  float32: {tiny_grad}")
print(f"  float16: {np.float16(tiny_grad)}")  # May become 0

# Loss scaling solution
loss_scale = 1024
scaled_grad = tiny_grad * loss_scale
print(f"\nLoss scaling (scale={loss_scale}):")
print(f"  Original: {tiny_grad} → float16: {np.float16(tiny_grad)}")
print(f"  Scaled:   {scaled_grad} → float16: {np.float16(scaled_grad)}")
print(f"  After unscale: {np.float16(scaled_grad) / loss_scale}")

---

## Summary: Key Takeaways

### Essential Concepts for ML Engineers

| Concept | Why It Matters | Action |
|---------|----------------|--------|
| **float32 vs float16** | Memory/speed tradeoff | Use mixed precision |
| **Machine epsilon** | Minimum useful learning rate | Check `np.finfo()` |
| **Overflow in softmax** | NaN in logits | Always subtract max |
| **Underflow in probs** | Zero probabilities | Work in log space |
| **Quantization** | Model deployment | PTQ or QAT |
| **Complex numbers** | Signal processing | FFT for audio/images |

### Best Practices

1. **Never compare floats with `==`** → Use `np.isclose()`
2. **Large logits?** → Use stable softmax
3. **Product of probabilities?** → Sum of log probs
4. **Deploying models?** → Consider int8 quantization
5. **Training large models?** → Use mixed precision (AMP)

---

## Practice Questions

Test your understanding:

1. **Why does `0.1 + 0.2 != 0.3` in Python?**

2. **What happens when you compute `np.exp(1000)`?**

3. **Why do ML frameworks subtract `max(x)` before computing softmax?**

4. **A model uses float32 weights. How much memory savings do you get with int8 quantization?**

5. **Why is BFloat16 often preferred over Float16 for training?**

---

### Next Steps

- **exercises.ipynb**: Practice problems with solutions
- **README.md**: Detailed theory and mathematical proofs
- Continue to: **02-Sets-and-Logic**